### Notebook for finding spatially proximate cameras
This notebook showcases how to find spatially proximate camera poses.

In [ ]:
import itertools
import sealoc.models as models

from loguru import logger
from sealoc.dal import create_data_access_layer, DataAccessLayer
from sealoc.geometry import CameraPoseIndex
from sealoc.geometry import create_camera_pose_index

#### Read camera bundle and poses

In [ ]:
dal: DataAccessLayer = create_data_access_layer(
    database_url="sqlite:////data/sealoc/sealoc.db",
)

with dal.session() as repos:
    camera_bundle: models.CameraBundle | None = repos.bundles.get_by_id(1)

    camera_ids: list[int] = [camera.id for camera in camera_bundle.cameras]

    camera_poses: list[models.CameraPose] = repos.poses.get_all_by_ids(camera_ids)
    camera_pose_map: dict[int, models.CameraPose] = {
        pose.camera_id: pose for pose in camera_poses
    }

    for camera in camera_bundle.cameras:
        if camera.id in camera_pose_map:
            camera.set_pose(camera_pose_map[camera.id])

cameras_with_pose: list[models.Camera] = [
    camera for camera in camera_bundle.cameras if camera.has_pose()
]
cameras_without_pose: list[models.Camera] = [
    camera for camera in camera_bundle.cameras if not camera.has_pose()
]

logger.info(f"Camera w/ pose:   {len(cameras_with_pose)}")
logger.info(f"Camera w/o pose:  {len(cameras_without_pose)}")

#### Create camera pose index

In [ ]:
import geopandas as gpd
import shapely

from tqdm.auto import tqdm


def find_pose_neighbors_for_group_pair(
    database_group: models.CameraGroup,
    query_group: models.CameraGroup,
) -> gpd.GeoDataFrame:
    database_cameras: dict[int, models.Camera] = {
        camera.id: camera for camera in database_group.cameras if camera.has_pose()
    }
    query_cameras: dict[int, models.Camera] = {
        camera.id: camera for camera in query_group.cameras if camera.has_pose()
    }

    logger.info(f"Database cameras: {len(database_cameras)}")
    logger.info(f"Query cameras:    {len(query_cameras)}")

    index: CameraPoseIndex = create_camera_pose_index(
        [camera.get_pose() for camera in database_cameras.values()]
    )

    reference_pose: models.CameraPose = list(query_cameras.values())[0].get_pose()

    rows: list[dict] = []
    for query_camera in tqdm(query_cameras.values(), desc="Finding camera neighbors..."):
        neighbors: list[models.CameraPose] = index.search(
            query_camera.get_pose(), max_distance=1.0
        )
        neighbor_cameras: list[models.Camera] = [
            database_cameras.get(pose.camera_id) for pose in neighbors
        ]
        new_rows: list[dict] = [
            _create_pose_link_from_cameras(query_camera, neighbor_camera)
            for neighbor_camera in neighbor_cameras
        ]
        rows.extend(new_rows)

    return gpd.GeoDataFrame(rows, crs=reference_pose.crs)


def _create_pose_link_from_cameras(
    query_camera: models.Camera,
    database_camera: models.Camera,
) -> dict:
    query_pose: models.CameraPose = query_camera.get_pose()
    database_pose: models.CameraPose = database_camera.get_pose()
    line_string: shapely.LineString = shapely.LineString(
        [query_pose.shape, database_pose.shape]
    )
    return {
        "query_camera_id": query_camera.id,
        "query_camera_label": query_camera.label,
        "database_camera_id": database_camera.id,
        "database_camera_label": database_camera.label,
        "geometry": line_string,
    }


camera_group_pairs: list[tuple] = list(itertools.combinations(camera_bundle.groups, 2))
camera_group_pair: tuple[models.CameraGroup, models.CameraGroup] = camera_group_pairs[0]

link_frame: gpd.GeoDataFrame = find_pose_neighbors_for_group_pair(
    database_group=camera_group_pair[0],
    query_group=camera_group_pair[1],
)

In [ ]:
logger.info(link_frame.head())
link_frame.to_file("~/output/link_frame.geojson")